# Project_1

In [2]:

import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
file_path = 'Dataset for Data Analytics_1_Raw.xlsx'
df = pd.read_excel(file_path)



In [3]:
df

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ORD201195,2024-06-20,C21126,Desk,1,107.04,392 Main St,Credit Card,Cancelled,TRK38009181,6,FREESHIP,Google,107.04
1196,ORD201196,2024-03-04,C20095,Monitor,2,662.53,778 Main St,Online,Cancelled,TRK69207593,5,NaN,Facebook,1325.06
1197,ORD201197,2023-07-13,C79674,Tablet,2,436.84,275 Main St,Online,Delivered,TRK88039356,2,FREESHIP,Instagram,873.68
1198,ORD201198,2024-08-22,C64753,Chair,4,262.52,509 Main St,Debit Card,Cancelled,TRK71683331,4,WINTER15,Instagram,1050.08


In [4]:
print("=" * 70)
print("DECODELABS PROJECT 1: DATA CLEANING & PREPARATION")
print("=" * 70)
print(f"\n RAW DATASET PROFILE")
print(f"   Rows: {df.shape[0]:,}")
print(f"   Columns: {df.shape[1]}")
print(f"   Column Names: {list(df.columns)}")

# Check for header row duplication (Row 1 is literally the headers repeated)
print(f"\n🔍 FIRST ROW CHECK (Potential Header Duplication):")
print(df.iloc[0].to_dict())

DECODELABS PROJECT 1: DATA CLEANING & PREPARATION

 RAW DATASET PROFILE
   Rows: 1,200
   Columns: 14
   Column Names: ['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice', 'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber', 'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice']

🔍 FIRST ROW CHECK (Potential Header Duplication):
{'OrderID': 'ORD200000', 'Date': Timestamp('2023-01-04 00:00:00'), 'CustomerID': 'C72649', 'Product': 'Monitor', 'Quantity': 5, 'UnitPrice': 570.62, 'ShippingAddress': '928 Main St', 'PaymentMethod': 'Debit Card', 'OrderStatus': 'Shipped', 'TrackingNumber': 'TRK37947903', 'ItemsInCart': 7, 'CouponCode': 'SAVE10', 'ReferralSource': 'Instagram', 'TotalPrice': 2853.1}


In [5]:
print(df.isna().sum())
# checking datatypes of columns
print(f"\n {df.dtypes}")

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

 OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object


## Detecting and Cleaning Dirty Data

### 1. Check for duplicate OrderIDs

In [8]:
# 1. Check for duplicate OrderIDs
print(f"\n DUPLICATE OrderIDs")
dup_orderids = df[df.duplicated(subset=['OrderID'], keep=False)].sort_values('OrderID')
print(f"   Total duplicate OrderID rows: {len(dup_orderids)}")
print(f"   Unique OrderIDs with duplicates: {dup_orderids['OrderID'].nunique()}")
if len(dup_orderids) > 0:
    print(f"\n   Sample duplicates:")
    print(dup_orderids[['OrderID', 'Date', 'CustomerID', 'Product', 'TotalPrice']].head(10).to_string(index=False))



 DUPLICATE OrderIDs
   Total duplicate OrderID rows: 0
   Unique OrderIDs with duplicates: 0


### 2. Check for nulls

In [10]:
# 2. Check for nulls
print(f"\n MISSING VALUES")
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) > 0:
    for col, count in nulls.items():
        pct = (count / len(df)) * 100
        print(f"   {col}: {count} missing ({pct:.1f}%)")
else:
    print("   No explicit nulls found")




 MISSING VALUES
   CouponCode: 309 missing (25.8%)


### Null Values Imputation

In [12]:
print("=" * 70)
print(" STRATEGIC IMPUTATION OF MISSING VALUES")
print("=" * 70)

# Handle CouponCode missing values (25.8% missing)
# Strategy: "No Coupon" is business-meaningful. no need to delete whole rows.
# Impute with "NONE" to preserve all 1,200 records.

missing_coupon_before = df['CouponCode'].isnull().sum()
print(f"\n Impute 'CouponCode' Missing Values")
print(f"   Before: {missing_coupon_before} missing ({missing_coupon_before/len(df)*100:.1f}%)")
print(f"   Strategy: Fill with 'NONE' (business-meaningful: no coupon used)")
print(f"     Warning from PDF: Listwise deletion reduces statistical power")
print(f"   Action: PRESERVE all 1,200 records")

df['CouponCode'] = df['CouponCode'].fillna('NONE')
missing_coupon_after = df['CouponCode'].isnull().sum()
print(f"   After: {missing_coupon_after} missing")
print(f"   Impact: Preserved 1,200 records (0% data loss)")

# Verify no other missing values remain
print(f"\n Post-Imputation Null Check:")
remaining_nulls = df.isnull().sum().sum()
print(f"   Total nulls in dataset: {remaining_nulls}")


 STRATEGIC IMPUTATION OF MISSING VALUES

 Impute 'CouponCode' Missing Values
   Before: 309 missing (25.8%)
   Strategy: Fill with 'NONE' (business-meaningful: no coupon used)
     Warning from PDF: Listwise deletion reduces statistical power
   Action: PRESERVE all 1,200 records
   After: 0 missing
   Impact: Preserved 1,200 records (0% data loss)

 Post-Imputation Null Check:
   Total nulls in dataset: 0


### 3. Check for empty strings / blank values

In [14]:
# 3. Check for empty strings / blank values
print(f"\n  BLANK/EMPTY VALUES")
for col in df.columns:
    if df[col].dtype == object:
        blanks = (df[col].astype(str).str.strip() == '').sum()
        if blanks > 0:
            print(f"   {col}: {blanks} blank values")
        else:
            print("No blanks/empty values")


  BLANK/EMPTY VALUES
No blanks/empty values
No blanks/empty values
No blanks/empty values
No blanks/empty values
No blanks/empty values
No blanks/empty values
No blanks/empty values
No blanks/empty values
No blanks/empty values


### 4.Date format issues

In [16]:
print("=" * 70)
print("Date format issues")
print("=" * 70)

print(f"\n  DATE FORMAT")
print(f"   Current Date type: {df['Date'].dtype}")
print(f"   Sample dates: {df['Date'].head(5).tolist()}")
# Check if dates are Excel serial numbers (integers)
if df['Date'].dtype in ['int64', 'float64']:
    print(f"     Dates are Excel serial numbers! Need conversion.")
else:
    print(f"   Dates appear to be datetime objects already. but need to be converted to ISO 8601 Dates (YYYY-MM-DD)")

Date format issues

  DATE FORMAT
   Current Date type: datetime64[ns]
   Sample dates: [Timestamp('2023-01-04 00:00:00'), Timestamp('2024-08-23 00:00:00'), Timestamp('2024-02-27 00:00:00'), Timestamp('2023-10-15 00:00:00'), Timestamp('2025-05-08 00:00:00')]
   Dates appear to be datetime objects already. but need to be converted to ISO 8601 Dates (YYYY-MM-DD)


### Date Format Standardization (ISO 8601: YYYY-MM-DD)

In [18]:
# Date Format Standardization (ISO 8601: YYYY-MM-DD)
print(f"\n  Date Format Standardization")
print(f"   Standard: ISO 8601 (YYYY-MM-DD)")
print(f"   Before: {df['Date'].dtype} | Sample: {df['Date'].iloc[0]}")

# Convert to ISO 8601 string format
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')
print(f"   After: string | Sample: {df['Date'].iloc[0]}")

# Validate no incorrectly formatted dates
invalid_dates = df[~df['Date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)]
print(f"   Invalid date formats: {len(invalid_dates)}")


  Date Format Standardization
   Standard: ISO 8601 (YYYY-MM-DD)
   Before: datetime64[ns] | Sample: 2023-01-04 00:00:00
   After: string | Sample: 2023-01-04
   Invalid date formats: 0


### 5.Numeric precision issues (floating point artifacts)

In [20]:
#  Numeric precision issues (floating point artifacts)
print(f"\n  FLOATING POINT PRECISION ARTIFACTS")
# Check TotalPrice for values with many decimals
df['TotalPrice_str'] = df['TotalPrice'].astype(str)
long_decimals = df[df['TotalPrice_str'].str.contains(r'\.\d{3,}')]
print(f"   TotalPrice values with >2 decimals: {len(long_decimals)}")
if len(long_decimals) > 0:
    print(f"   Examples: {long_decimals['TotalPrice'].head(5).tolist()}")


  FLOATING POINT PRECISION ARTIFACTS
   TotalPrice values with >2 decimals: 29
   Examples: [769.3799999999999, 635.9000000000001, 440.1899999999999, 927.6600000000001, 950.4000000000001]


In [21]:
# 5: Numeric Precision (2 decimals for monetary fields)
print(f"\n  Numeric Precision (2 decimals)")
print(f"   Before: Floating point artifacts like 769.3799999999999")

df['UnitPrice'] = df['UnitPrice'].round(2)
df['TotalPrice'] = df['TotalPrice'].round(2)

# Verify
long_decimals_after = df[df['TotalPrice'].astype(str).str.contains(r'\.\d{3,}')]
print(f"   After: Values with >2 decimals: {len(long_decimals_after)}")
print(f"   Sample corrected: {df['TotalPrice'].iloc[38]} (was 769.3799999999999)")


  Numeric Precision (2 decimals)
   Before: Floating point artifacts like 769.3799999999999
   After: Values with >2 decimals: 0
   Sample corrected: 245.89 (was 769.3799999999999)


### 6.Text Standardization (Proper Case & Trim Whitespace)

In [23]:
# 3C: Text Standardization (Proper Case & Trim Whitespace)
print(f"\n Text Standardization")
text_cols = ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource', 'ShippingAddress']

for col in text_cols:
    # Trim whitespace
    df[col] = df[col].astype(str).str.strip()
    # Proper case for text fields
    if col in ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource']:
        df[col] = df[col].str.title()

print(f"   Applied: Trim Whitespace + Proper Case")
print(f"   Product values: {sorted(df['Product'].unique())}")
print(f"   PaymentMethod values: {sorted(df['PaymentMethod'].unique())}")


 Text Standardization
   Applied: Trim Whitespace + Proper Case
   Product values: ['Chair', 'Desk', 'Laptop', 'Monitor', 'Phone', 'Printer', 'Tablet']
   PaymentMethod values: ['Cash', 'Credit Card', 'Debit Card', 'Gift Card', 'Online']


### 7. Computed validation: Quantity * UnitPrice vs TotalPrice

In [25]:
#  Computed Field Validation
print(f"\n Computed Field Validation")
df['ComputedTotal'] = (df['Quantity'] * df['UnitPrice']).round(2)
df['PriceDiff'] = abs(df['TotalPrice'] - df['ComputedTotal'])
mismatches = df[df['PriceDiff'] > 0.01]
print(f"   TotalPrice = Quantity × UnitPrice validation")
print(f"   Mismatches after rounding: {len(mismatches)}")
if len(mismatches) == 0:
    print(f"    All computed totals validate correctly")


 Computed Field Validation
   TotalPrice = Quantity × UnitPrice validation
   Mismatches after rounding: 0
    All computed totals validate correctly


### 8. Check for inconsistent text

In [27]:
# 8. Check for inconsistent text
print(f"\n TEXT STANDARDIZATION")
print(f"   Product categories: {sorted(df['Product'].unique())}")
print(f"   PaymentMethods: {sorted(df['PaymentMethod'].unique())}")
print(f"   OrderStatus values: {sorted(df['OrderStatus'].unique())}")
print(f"   ReferralSources: {sorted(df['ReferralSource'].unique())}")


 TEXT STANDARDIZATION
   Product categories: ['Chair', 'Desk', 'Laptop', 'Monitor', 'Phone', 'Printer', 'Tablet']
   PaymentMethods: ['Cash', 'Credit Card', 'Debit Card', 'Gift Card', 'Online']
   OrderStatus values: ['Cancelled', 'Delivered', 'Pending', 'Returned', 'Shipped']
   ReferralSources: ['Email', 'Facebook', 'Google', 'Instagram', 'Referral']


In [28]:
df.to_excel('Dataset_Cleaned.xlsx', index=False)